# Local RAG Evaluation

Run the FastAPI application first. This notebook uses its actual local embedding, FAISS retrieval, and Hugging Face generation path. It does not call OpenAI or use prefilled evaluation results.

In [ ]:
import json
import os
from pathlib import Path
import httpx

from IPython.display import Markdown, display

BASE_URL = os.environ.get('RAG_API_URL', 'http://127.0.0.1:8000')
ROOT = Path.cwd().parent if Path.cwd().name == 'evaluation' else Path.cwd()
SAMPLE_PATH = ROOT / 'data' / 'sample_documents' / 'rag_fundamentals.txt'
QUESTIONS_PATH = ROOT / 'evaluation' / 'questions.json'
questions = json.loads(QUESTIONS_PATH.read_text(encoding='utf-8'))
assert len(questions) == 10, f'Expected exactly 10 questions, got {len(questions)}'
health = httpx.get(f'{BASE_URL}/api/health', timeout=10)
health.raise_for_status()
print(f"API status: {health.json()['status']}; questions: {len(questions)}; endpoint: {BASE_URL}")

## Load the sample document

The document is uploaded only when it is not already present in the library.

In [ ]:
documents_response = httpx.get(f'{BASE_URL}/api/documents', timeout=20)
documents_response.raise_for_status()
documents = documents_response.json()
if not any(item['filename'] == SAMPLE_PATH.name for item in documents):
    with SAMPLE_PATH.open('rb') as sample_file:
        upload = httpx.post(
            f'{BASE_URL}/api/documents/upload',
            files={'file': (SAMPLE_PATH.name, sample_file, 'text/plain')},
            timeout=180,
        )
    upload.raise_for_status()
    print(f"Uploaded {upload.json()['filename']} ({upload.json()['chunk_count']} chunks)")
else:
    print(f'{SAMPLE_PATH.name} is already indexed')

## Run all questions

Answers and citations below are returned by the running application. Retrieval relevance is a hit when the expected filename and page occur among the top five sources.

In [ ]:
results = []
for number, item in enumerate(questions, start=1):
    response = httpx.post(
        f'{BASE_URL}/api/ask',
        json={'question': item['question']},
        timeout=180,
    )
    response.raise_for_status()
    payload = response.json()
    sources = payload['sources'][:5]
    relevant = any(
        source['filename'] == item['expected_source']
        and (item.get('expected_page') is None or source['page_number'] == item['expected_page'])
        for source in sources
    )
    result = {
        'question': item['question'],
        'expected_answer': item['expected_answer'],
        'answer': payload['answer'],
        'retrieval_relevant': relevant,
        'sources': sources,
    }
    results.append(result)
    print(f'\n{number}. {item["question"]}')
    print(f'Expected: {item["expected_answer"]}')
    print(f'Generated: {payload["answer"]}')
    print(f'Retrieval relevant@5: {relevant}')
    print('Sources:', [(source['filename'], source['page_number'], source['chunk_index'], round(source['score'], 3)) for source in sources])
assert len(results) == 10

## Review and score

Classify each displayed answer against its expected answer. The notebook calculates accuracy from these reviewer labels and displays the actual returned citations.

In [ ]:
labels = {'correct': 'Correct', 'partially correct': 'Partially Correct', 'incorrect': 'Incorrect'}
for number, result in enumerate(results, start=1):
    while True:
        grade = input(f'{number}/10 answer grade (Correct / Partially Correct / Incorrect): ').strip().lower()
        if grade in labels:
            result['answer_grade'] = labels[grade]
            break
        print('Enter Correct, Partially Correct, or Incorrect.')

total = len(results)
correct = sum(result['answer_grade'] == 'Correct' for result in results)
partial = sum(result['answer_grade'] == 'Partially Correct' for result in results)
incorrect = sum(result['answer_grade'] == 'Incorrect' for result in results)
retrieval_hits = sum(result['retrieval_relevant'] for result in results)
answer_accuracy = 100 * correct / total
retrieval_relevance = 100 * retrieval_hits / total
print(f'Answer accuracy: {answer_accuracy:.1f}% ({correct}/{total} fully correct)')
print(f'Retrieval relevance@5: {retrieval_relevance:.1f}% ({retrieval_hits}/{total})')

def table_text(value):
    return str(value).replace('|', '\\|').replace('\n', ' ')

rows = [
    '| # | Question | Expected answer | Generated answer | Grade | Relevant@5 | Sources |',
    '|---:|---|---|---|---|:---:|---|',
]
for number, result in enumerate(results, start=1):
    source_text = '; '.join(
        f"{source['filename']} p.{source['page_number']} c.{source['chunk_index']} ({source['score']:.3f})"
        for source in result['sources']
    )
    rows.append(
        f"| {number} | {table_text(result['question'])} | {table_text(result['expected_answer'])} "
        f"| {table_text(result['answer'])} | {result['answer_grade']} "
        f"| {'Yes' if result['retrieval_relevant'] else 'No'} | {table_text(source_text)} |"
    )
display(Markdown('\n'.join(rows)))